In [ ]:
"""
=================================================================
DERMASCAN-AI — PHASE 1: DATA PIPELINE
=================================================================
"""

import os
import json
import shutil
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("✅ Imports done")

In [ ]:
"""
=================================================================
PATHS & CLASS CONFIG
=================================================================
"""

# ── INPUT PATHS ──
HAM_BASE     = Path("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000")
HAM_METADATA = HAM_BASE / "HAM10000_metadata.csv"
HAM_IMG_DIRS = [
    HAM_BASE / "HAM10000_images_part_1",
    HAM_BASE / "HAM10000_images_part_2",
]

DERMNET_TRAIN = Path("/kaggle/input/datasets/shubhamgoel27/dermnet/train")
DERMNET_TEST  = Path("/kaggle/input/datasets/shubhamgoel27/dermnet/test")

# ── OUTPUT PATHS ──
OUTPUT_DIR    = Path("/kaggle/working")
PROCESSED_DIR = OUTPUT_DIR / "processed"
SPLITS_DIR    = OUTPUT_DIR / "splits"
RESULTS_DIR   = OUTPUT_DIR / "results"

for d in [PROCESSED_DIR, SPLITS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── 13-CLASS SYSTEM ──
CLASS_CONFIG = {
    0:  {"name": "Melanoma",              "folder": "Melanoma",              "tier": "CANCER",     "severity": "CRITICAL", "color": "#e74c3c"},
    1:  {"name": "Basal Cell Carcinoma",  "folder": "Basal_Cell_Carcinoma",  "tier": "CANCER",     "severity": "HIGH",     "color": "#e67e22"},
    2:  {"name": "Actinic Keratoses",     "folder": "Actinic_Keratoses",     "tier": "PRE-CANCER", "severity": "MEDIUM",   "color": "#f39c12"},
    3:  {"name": "Melanocytic Nevi",      "folder": "Melanocytic_Nevi",      "tier": "BENIGN",     "severity": "LOW",      "color": "#27ae60"},
    4:  {"name": "Benign Keratosis",      "folder": "Benign_Keratosis",      "tier": "BENIGN",     "severity": "LOW",      "color": "#2ecc71"},
    5:  {"name": "Dermatofibroma",        "folder": "Dermatofibroma",        "tier": "BENIGN",     "severity": "LOW",      "color": "#1abc9c"},
    6:  {"name": "Vascular Lesions",      "folder": "Vascular_Lesions",      "tier": "BENIGN",     "severity": "LOW",      "color": "#16a085"},
    7:  {"name": "Acne and Rosacea",      "folder": "Acne",                  "tier": "DISEASE",    "severity": "LOW",      "color": "#3498db"},
    8:  {"name": "Eczema",                "folder": "Eczema",                "tier": "DISEASE",    "severity": "LOW",      "color": "#2980b9"},
    9:  {"name": "Psoriasis",             "folder": "Psoriasis",             "tier": "DISEASE",    "severity": "LOW",      "color": "#9b59b6"},
    10: {"name": "Fungal Infection",      "folder": "Fungal_Infection",      "tier": "DISEASE",    "severity": "LOW",      "color": "#8e44ad"},
    11: {"name": "Warts and Viral",       "folder": "Warts",                 "tier": "DISEASE",    "severity": "LOW",      "color": "#34495e"},
    12: {"name": "Vitiligo",              "folder": "Vitiligo",              "tier": "DISEASE",    "severity": "LOW",      "color": "#7f8c8d"},
}

CLASS_NAMES    = {k: v["name"] for k, v in CLASS_CONFIG.items()}
CLASS_FOLDERS  = {k: v["folder"] for k, v in CLASS_CONFIG.items()}
FOLDER_TO_IDX  = {v["folder"]: k for k, v in CLASS_CONFIG.items()}

# HAM10000 dx code → our folder
HAM_TO_CLASS = {
    "mel": "Melanoma", "bcc": "Basal_Cell_Carcinoma", "akiec": "Actinic_Keratoses",
    "nv": "Melanocytic_Nevi", "bkl": "Benign_Keratosis", "df": "Dermatofibroma", "vasc": "Vascular_Lesions",
}

# DermNet folder substring → our folder
DERMNET_MATCH_RULES = {
    "Acne":              ["acne", "rosacea"],
    "Eczema":            ["eczema", "atopic dermatitis"],
    "Psoriasis":         ["psoriasis", "lichen planus"],
    "Fungal_Infection":  ["tinea", "ringworm", "candidiasis", "fungal"],
    "Warts":             ["warts", "molluscum", "viral"],
    "Vitiligo":          ["vitiligo", "pigmentation", "light diseases"],
}

# ── VERIFY ──
print(f"HAM metadata exists  : {HAM_METADATA.exists()}")
print(f"HAM images part1     : {HAM_IMG_DIRS[0].exists()}")
print(f"HAM images part2     : {HAM_IMG_DIRS[1].exists()}")
print(f"DermNet train exists : {DERMNET_TRAIN.exists()}")
print(f"DermNet test exists  : {DERMNET_TEST.exists()}")
print(f"\n✅ 13 classes configured")

In [ ]:
"""
=================================================================
EXPLORE DERMNET & BUILD MAPPING
=================================================================
"""

print("DermNet categories:\n")

dermnet_mapping = {}  # actual_folder_name → our_class

for split_dir in [DERMNET_TRAIN, DERMNET_TEST]:
    for cat_dir in sorted(split_dir.iterdir()):
        if not cat_dir.is_dir():
            continue
        cat_name = cat_dir.name
        if cat_name in dermnet_mapping:
            continue
        
        # Try matching
        cat_lower = cat_name.lower()
        matched = None
        for our_class, keywords in DERMNET_MATCH_RULES.items():
            if any(kw in cat_lower for kw in keywords):
                matched = our_class
                break
        
        dermnet_mapping[cat_name] = matched
        
        n_imgs = len([f for f in cat_dir.iterdir() if f.suffix.lower() in ['.jpg','.jpeg','.png']])
        tag = f"✅ → {matched}" if matched else "⬜ SKIP"
        print(f"  [{n_imgs:5d}]  {cat_name:<55s} {tag}")

print(f"\nMapped categories: {sum(1 for v in dermnet_mapping.values() if v)}")
print(f"Skipped categories: {sum(1 for v in dermnet_mapping.values() if not v)}")

In [ ]:
"""
=================================================================
PROCESS HAM10000
=================================================================
"""

print("🔧 Processing HAM10000...\n")

ham_df = pd.read_csv(HAM_METADATA)
print(f"Metadata: {len(ham_df)} rows, columns: {list(ham_df.columns)}")

# Build image_id → path lookup
ham_image_paths = {}
for img_dir in HAM_IMG_DIRS:
    for img_path in img_dir.glob("*.jpg"):
        ham_image_paths[img_path.stem] = img_path

print(f"Total images found: {len(ham_image_paths)}")

# Create folders & copy
for folder in HAM_TO_CLASS.values():
    (PROCESSED_DIR / folder).mkdir(parents=True, exist_ok=True)

ham_records = []
missing = 0

for _, row in tqdm(ham_df.iterrows(), total=len(ham_df), desc="HAM10000"):
    img_id = row['image_id']
    dx = row['dx']
    
    if dx not in HAM_TO_CLASS:
        continue
    
    src = ham_image_paths.get(img_id)
    if src is None:
        missing += 1
        continue
    
    folder = HAM_TO_CLASS[dx]
    dst = PROCESSED_DIR / folder / f"{img_id}.jpg"
    
    if not dst.exists():
        shutil.copy2(src, dst)
    
    ham_records.append({
        'image_id': img_id,
        'filename': f"{img_id}.jpg",
        'class_folder': folder,
        'class_name': CLASS_CONFIG[FOLDER_TO_IDX[folder]]['name'],
        'class_idx': FOLDER_TO_IDX[folder],
        'source': 'ham10000',
        'lesion_id': row['lesion_id'],
        'age': row.get('age'), 'sex': row.get('sex'),
        'localization': row.get('localization'),
    })

ham_records_df = pd.DataFrame(ham_records)
print(f"\n✅ Processed: {len(ham_records_df)} | Missing: {missing}")
print(ham_records_df['class_folder'].value_counts().to_string())

In [ ]:
"""
=================================================================
PROCESS DERMNET
=================================================================
"""

print("🔧 Processing DermNet...\n")

for folder in set(v for v in dermnet_mapping.values() if v):
    (PROCESSED_DIR / folder).mkdir(parents=True, exist_ok=True)

dermnet_records = []
copied = 0
skipped = 0
errors = 0

for split_dir in [DERMNET_TRAIN, DERMNET_TEST]:
    for cat_dir in sorted(split_dir.iterdir()):
        if not cat_dir.is_dir():
            continue
        
        our_class = dermnet_mapping.get(cat_dir.name)
        if our_class is None:
            skipped += len(list(cat_dir.iterdir()))
            continue
        
        for img_path in cat_dir.iterdir():
            if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png', '.bmp']:
                continue
            
            unique_name = f"dn_{our_class}_{img_path.stem}{img_path.suffix}"
            dst = PROCESSED_DIR / our_class / unique_name
            
            try:
                if not dst.exists():
                    shutil.copy2(img_path, dst)
                
                dermnet_records.append({
                    'image_id': img_path.stem,
                    'filename': unique_name,
                    'class_folder': our_class,
                    'class_name': CLASS_CONFIG[FOLDER_TO_IDX[our_class]]['name'],
                    'class_idx': FOLDER_TO_IDX[our_class],
                    'source': 'dermnet',
                    'lesion_id': f"dn_{img_path.stem}",
                    'age': None, 'sex': None, 'localization': None,
                })
                copied += 1
            except:
                errors += 1

dermnet_records_df = pd.DataFrame(dermnet_records)
print(f"✅ Copied: {copied} | Skipped: {skipped} | Errors: {errors}")
if len(dermnet_records_df) > 0:
    print(dermnet_records_df['class_folder'].value_counts().to_string())

In [ ]:
"""
=================================================================
COMBINE INTO MASTER DATAFRAME
=================================================================
"""

# Add missing columns so concat works
for col in ['dermnet_category', 'dermnet_split']:
    if col not in ham_records_df.columns:
        ham_records_df[col] = None
    if col not in dermnet_records_df.columns:
        dermnet_records_df[col] = None

master_df = pd.concat([ham_records_df, dermnet_records_df], ignore_index=True)

# Add full path
master_df['image_path'] = master_df.apply(
    lambda r: str(PROCESSED_DIR / r['class_folder'] / r['filename']), axis=1
)

# Add tier & severity
tier_map = {v['folder']: v['tier'] for v in CLASS_CONFIG.values()}
sev_map  = {v['folder']: v['severity'] for v in CLASS_CONFIG.values()}
master_df['tier'] = master_df['class_folder'].map(tier_map)
master_df['severity'] = master_df['class_folder'].map(sev_map)
master_df['is_cancer'] = master_df['tier'].isin(['CANCER', 'PRE-CANCER']).astype(int)

print(f"📊 MASTER DATASET: {len(master_df):,} images")
print(f"   HAM10000: {len(master_df[master_df['source']=='ham10000']):,}")
print(f"   DermNet : {len(master_df[master_df['source']=='dermnet']):,}")
print(f"   Classes : {master_df['class_idx'].nunique()}")

print(f"\n{'CLASS':<30s} {'COUNT':>6s} {'%':>6s} {'TIER':>12s}")
print("─" * 60)
for idx in sorted(CLASS_CONFIG.keys()):
    info = CLASS_CONFIG[idx]
    n = len(master_df[master_df['class_idx'] == idx])
    print(f"{info['name']:<30s} {n:>6d} {n/len(master_df)*100:>5.1f}% {info['tier']:>12s}")
print("─" * 60)
print(f"{'TOTAL':<30s} {len(master_df):>6d}")

master_df.to_csv(OUTPUT_DIR / "master_dataset.csv", index=False)

In [ ]:
"""
=================================================================
CLEAN CORRUPTED / INVALID IMAGES
=================================================================
"""

print("🧹 Validating images...\n")

bad_indices = []

for idx, row in tqdm(master_df.iterrows(), total=len(master_df), desc="Validating"):
    try:
        img = Image.open(row['image_path'])
        img.verify()
        img = Image.open(row['image_path'])
        w, h = img.size
        if w < 32 or h < 32:
            bad_indices.append(idx)
    except:
        bad_indices.append(idx)

print(f"\n✅ Valid  : {len(master_df) - len(bad_indices):,}")
print(f"❌ Bad    : {len(bad_indices)}")

if bad_indices:
    master_df = master_df.drop(bad_indices).reset_index(drop=True)
    print(f"🗑️ Removed {len(bad_indices)} images → {len(master_df):,} remaining")

master_df.to_csv(OUTPUT_DIR / "master_dataset.csv", index=False)

In [ ]:
"""
=================================================================
DOWNSAMPLE MELANOCYTIC NEVI (6700 → 2000)
=================================================================
"""

print("⚖️ Balancing classes...\n")

NV_TARGET = 2000
nv_mask = master_df['class_folder'] == 'Melanocytic_Nevi'
nv_df = master_df[nv_mask]
other_df = master_df[~nv_mask]

print(f"Before: nv = {len(nv_df)}")

if len(nv_df) > NV_TARGET:
    nv_lesions = nv_df['lesion_id'].unique()
    np.random.shuffle(nv_lesions)
    
    keep = []
    for lesion in nv_lesions:
        keep.append(nv_df[nv_df['lesion_id'] == lesion])
        if sum(len(x) for x in keep) >= NV_TARGET:
            break
    
    nv_downsampled = pd.concat(keep).head(NV_TARGET)
    master_df = pd.concat([other_df, nv_downsampled], ignore_index=True)

print(f"After : nv = {len(master_df[master_df['class_folder']=='Melanocytic_Nevi'])}")
print(f"Total : {len(master_df):,}")

counts = master_df['class_folder'].value_counts()
print(f"Imbalance ratio: {counts.max()/counts.min():.1f}x")

master_df.to_csv(OUTPUT_DIR / "master_dataset.csv", index=False)

In [ ]:
"""
=================================================================
EDA: CLASS DISTRIBUTION
=================================================================
"""

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Bar chart
class_data = []
for idx in sorted(CLASS_CONFIG.keys()):
    info = CLASS_CONFIG[idx]
    count = len(master_df[master_df['class_idx'] == idx])
    class_data.append({'name': info['name'], 'count': count, 'color': info['color'], 'tier': info['tier']})

class_data.sort(key=lambda x: x['count'], reverse=True)

ax = axes[0]
bars = ax.barh([d['name'] for d in class_data], [d['count'] for d in class_data],
               color=[d['color'] for d in class_data], edgecolor='white')
for bar, d in zip(bars, class_data):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f"{d['count']:,}", va='center', fontweight='bold', fontsize=10)
ax.set_xlabel('Number of Images')
ax.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax.invert_yaxis()

# Tier pie
ax = axes[1]
tier_counts = master_df['tier'].value_counts()
tier_colors = {'CANCER': '#e74c3c', 'PRE-CANCER': '#f39c12', 'BENIGN': '#27ae60', 'DISEASE': '#3498db'}
ax.pie(tier_counts.values, labels=[f"{t}\n({c:,})" for t, c in tier_counts.items()],
       colors=[tier_colors.get(t, '#95a5a6') for t in tier_counts.index],
       autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11}, explode=[0.05]*len(tier_counts))
ax.set_title('Distribution by Tier', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(RESULTS_DIR / "class_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
"""
=================================================================
EDA: SAMPLE IMAGES FROM ALL 13 CLASSES
=================================================================
"""

fig = plt.figure(figsize=(24, 36))
fig.suptitle('Sample Images — All 13 Classes', fontsize=20, fontweight='bold', y=1.01)
gs = gridspec.GridSpec(13, 5, hspace=0.4, wspace=0.1)

for class_idx in sorted(CLASS_CONFIG.keys()):
    info = CLASS_CONFIG[class_idx]
    class_data = master_df[master_df['class_idx'] == class_idx]
    
    n_samples = min(5, len(class_data))
    if n_samples == 0:
        continue
    samples = class_data.sample(n_samples, random_state=SEED)
    
    for j, (_, row) in enumerate(samples.iterrows()):
        ax = fig.add_subplot(gs[class_idx, j])
        try:
            img = Image.open(row['image_path'])
            ax.imshow(img)
        except:
            ax.text(0.5, 0.5, 'Error', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')
        
        if j == 0:
            tier_emoji = {"CANCER": "🔴", "PRE-CANCER": "🟡", "BENIGN": "🟢", "DISEASE": "🔵"}
            ax.set_title(f"{tier_emoji.get(info['tier'],'')} {info['name']}\n({len(class_data)} imgs)",
                        fontsize=9, fontweight='bold', loc='left', color=info['color'])

plt.savefig(RESULTS_DIR / "sample_images.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
"""
=================================================================
EDA: IMAGE DIMENSIONS & SOURCE ANALYSIS
=================================================================
"""

sample = master_df.sample(min(2000, len(master_df)), random_state=SEED)
dims = []

for _, row in tqdm(sample.iterrows(), total=len(sample), desc="Analyzing"):
    try:
        img = Image.open(row['image_path'])
        w, h = img.size
        dims.append({'w': w, 'h': h, 'source': row['source']})
    except:
        pass

dims_df = pd.DataFrame(dims)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(dims_df['w'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Width Distribution', fontweight='bold')
axes[0].set_xlabel('Pixels')

axes[1].hist(dims_df['h'], bins=50, color='coral', edgecolor='white')
axes[1].set_title('Height Distribution', fontweight='bold')
axes[1].set_xlabel('Pixels')

# Source breakdown
ax = axes[2]
source_class = master_df.groupby(['class_folder', 'source']).size().unstack(fill_value=0)
source_class.plot(kind='barh', stacked=True, ax=ax, color=['#3498db', '#e74c3c'])
ax.set_title('Images by Source', fontweight='bold')
ax.set_xlabel('Count')

plt.tight_layout()
plt.savefig(RESULTS_DIR / "image_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Width  — Min: {dims_df['w'].min()}, Max: {dims_df['w'].max()}, Median: {dims_df['w'].median():.0f}")
print(f"Height — Min: {dims_df['h'].min()}, Max: {dims_df['h'].max()}, Median: {dims_df['h'].median():.0f}")

In [ ]:
"""
=================================================================
CREATE TRAIN / VAL / TEST SPLITS
⚠️ HAM10000: split by lesion_id (prevent data leakage)
   DermNet:  stratified random split
=================================================================
"""

print("✂️ Creating splits...\n")

# ── SPLIT HAM10000 (by lesion_id) ──
ham_data = master_df[master_df['source'] == 'ham10000'].copy().reset_index(drop=True)

sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
trainval_idx, test_idx = next(sgkf.split(ham_data, ham_data['class_idx'], ham_data['lesion_id']))
ham_trainval = ham_data.iloc[trainval_idx].reset_index(drop=True)
ham_test = ham_data.iloc[test_idx].reset_index(drop=True)

sgkf2 = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
train_idx, val_idx = next(sgkf2.split(ham_trainval, ham_trainval['class_idx'], ham_trainval['lesion_id']))
ham_train = ham_trainval.iloc[train_idx].reset_index(drop=True)
ham_val = ham_trainval.iloc[val_idx].reset_index(drop=True)

# Verify NO leakage
assert len(set(ham_train['lesion_id']) & set(ham_val['lesion_id'])) == 0, "LEAKAGE!"
assert len(set(ham_train['lesion_id']) & set(ham_test['lesion_id'])) == 0, "LEAKAGE!"
assert len(set(ham_val['lesion_id']) & set(ham_test['lesion_id'])) == 0, "LEAKAGE!"
print(f"✅ HAM10000 — NO LEAKAGE")
print(f"   Train: {len(ham_train)} | Val: {len(ham_val)} | Test: {len(ham_test)}")

# ── SPLIT DERMNET (stratified) ──
derm_data = master_df[master_df['source'] == 'dermnet'].copy().reset_index(drop=True)

skf = StratifiedKFold(n_splits=7, shuffle=True, random_state=SEED)
trainval_idx, test_idx = next(skf.split(derm_data, derm_data['class_idx']))
derm_trainval = derm_data.iloc[trainval_idx].reset_index(drop=True)
derm_test = derm_data.iloc[test_idx].reset_index(drop=True)

skf2 = StratifiedKFold(n_splits=6, shuffle=True, random_state=SEED)
train_idx, val_idx = next(skf2.split(derm_trainval, derm_trainval['class_idx']))
derm_train = derm_trainval.iloc[train_idx].reset_index(drop=True)
derm_val = derm_trainval.iloc[val_idx].reset_index(drop=True)

print(f"✅ DermNet")
print(f"   Train: {len(derm_train)} | Val: {len(derm_val)} | Test: {len(derm_test)}")

# ── COMBINE ──
train_df = pd.concat([ham_train, derm_train], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df   = pd.concat([ham_val, derm_val], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df  = pd.concat([ham_test, derm_test], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"\n📊 FINAL SPLITS:")
print(f"   Train : {len(train_df):,} ({len(train_df)/(len(train_df)+len(val_df)+len(test_df))*100:.1f}%)")
print(f"   Val   : {len(val_df):,} ({len(val_df)/(len(train_df)+len(val_df)+len(test_df))*100:.1f}%)")
print(f"   Test  : {len(test_df):,} ({len(test_df)/(len(train_df)+len(val_df)+len(test_df))*100:.1f}%)")

In [ ]:
"""
=================================================================
VERIFY & SAVE
=================================================================
"""

print("✅ Class distribution per split:\n")
print(f"{'CLASS':<30s} {'TRAIN':>7s} {'VAL':>7s} {'TEST':>7s} {'TOTAL':>7s}")
print("─" * 65)

for idx in sorted(CLASS_CONFIG.keys()):
    info = CLASS_CONFIG[idx]
    folder = info['folder']
    nt = len(train_df[train_df['class_folder'] == folder])
    nv = len(val_df[val_df['class_folder'] == folder])
    ns = len(test_df[test_df['class_folder'] == folder])
    print(f"{info['name']:<30s} {nt:>7d} {nv:>7d} {ns:>7d} {nt+nv+ns:>7d}")

print("─" * 65)
print(f"{'TOTAL':<30s} {len(train_df):>7d} {len(val_df):>7d} {len(test_df):>7d} {len(train_df)+len(val_df)+len(test_df):>7d}")

# Verify all paths exist
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    missing = sum(1 for _, r in df.iterrows() if not os.path.exists(r['image_path']))
    print(f"\n{'✅' if missing==0 else '❌'} {name}: {missing} missing")

# Save
save_cols = ['image_path', 'filename', 'class_folder', 'class_name', 'class_idx',
             'source', 'lesion_id', 'tier', 'severity', 'is_cancer']

train_df[save_cols].to_csv(SPLITS_DIR / "train.csv", index=False)
val_df[save_cols].to_csv(SPLITS_DIR / "val.csv", index=False)
test_df[save_cols].to_csv(SPLITS_DIR / "test.csv", index=False)

print(f"\n💾 Saved:")
print(f"   {SPLITS_DIR}/train.csv ({len(train_df):,} rows)")
print(f"   {SPLITS_DIR}/val.csv ({len(val_df):,} rows)")
print(f"   {SPLITS_DIR}/test.csv ({len(test_df):,} rows)")

In [ ]:
"""
=================================================================
SPLIT DISTRIBUTION VISUALIZATION
=================================================================
"""

fig, axes = plt.subplots(1, 3, figsize=(22, 8))
fig.suptitle('Train / Val / Test Distribution', fontsize=18, fontweight='bold')

for ax_idx, (name, sdf) in enumerate([("Train", train_df), ("Val", val_df), ("Test", test_df)]):
    ax = axes[ax_idx]
    names, counts, colors = [], [], []
    
    for idx in sorted(CLASS_CONFIG.keys()):
        info = CLASS_CONFIG[idx]
        n = len(sdf[sdf['class_idx'] == idx])
        names.append(info['name'])
        counts.append(n)
        colors.append(info['color'])
    
    ax.barh(names, counts, color=colors, edgecolor='white')
    for i, c in enumerate(counts):
        ax.text(c + 5, i, str(c), va='center', fontsize=9, fontweight='bold')
    ax.set_title(f'{name} ({len(sdf):,})', fontsize=14, fontweight='bold')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "split_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
"""
=================================================================
SAVE CONFIG & FINAL SUMMARY
=================================================================
"""

# Save class config JSON
config_save = {str(k): {kk: vv for kk, vv in v.items()} for k, v in CLASS_CONFIG.items()}
with open(OUTPUT_DIR / "class_config.json", 'w') as f:
    json.dump(config_save, f, indent=2)

total = len(train_df) + len(val_df) + len(test_df)

print("🎉" * 25)
print(f"""
{'='*60}
  PHASE 1 COMPLETE — DERMASCAN-AI DATA PIPELINE
{'='*60}

📊 Dataset:  {total:,} images across 13 classes
   HAM10000: {len(master_df[master_df['source']=='ham10000']):,} images (cancer/benign)
   DermNet:  {len(master_df[master_df['source']=='dermnet']):,} images (diseases)

✂️ Splits:
   Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}

🔒 Data Leakage: PREVENTED (HAM10000 split by lesion_id)

📁 Outputs:
   {SPLITS_DIR}/train.csv
   {SPLITS_DIR}/val.csv  
   {SPLITS_DIR}/test.csv
   {OUTPUT_DIR}/class_config.json
   {OUTPUT_DIR}/master_dataset.csv

📈 Plots:
   {RESULTS_DIR}/class_distribution.png
   {RESULTS_DIR}/sample_images.png
   {RESULTS_DIR}/image_analysis.png
   {RESULTS_DIR}/split_distribution.png

""")